# ⚽ Video → Anime — 1-click Colab
Upload a football MP4. The notebook installs the GPU runtime, downloads the project, verifies CUDA, renders with AnimeGANv2 + temporal stabilization, preserves source audio, then downloads the result.


In [ ]:
#@title 1. Setup GPU + project
import os, subprocess, sys
subprocess.run(['nvidia-smi'], check=True)
os.chdir('/content')
if os.path.exists('/content/video-to-animation'):
    subprocess.run(['rm','-rf','/content/video-to-animation'], check=True)
subprocess.run(['git','clone','https://github.com/v-tech-hub/video-to-animation.git'], check=True)
os.chdir('/content/video-to-animation')
subprocess.run([sys.executable,'-m','pip','uninstall','-y','onnxruntime','onnxruntime-gpu'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable,'-m','pip','install','-q','numpy<2.4','opencv-python-headless>=4.10','onnxruntime-gpu>=1.20'], check=True)
import onnxruntime as ort
print('Providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDA unavailable. In Colab choose Runtime > Change runtime type > GPU, then rerun.'


In [ ]:
#@title 2. Upload football video
from google.colab import files
uploaded = files.upload()
INPUT = next(iter(uploaded.keys()))
print('Input:', INPUT)


In [ ]:
#@title 3. Render anime video
INFERENCE_WIDTH = 960 #@param {type:'integer'}
TEMPORAL = 0.18 #@param {type:'number'}
OUTPUT = '/content/goal-anime.mp4'
cmd = [sys.executable,'football_cartoon.py',INPUT,'-o',OUTPUT,'--temporal',str(TEMPORAL),'--inference-width',str(INFERENCE_WIDTH)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('DONE:', OUTPUT)


In [ ]:
#@title 4. Preview + download
from IPython.display import Video, display
display(Video(OUTPUT, embed=True, width=960))
files.download(OUTPUT)
